# Alpha Connectome, sweep 7: training dynamics (9 GRUs) + TCN & minGRU teachers (4 models)

Small launcher: takes the base files from `connectome_sweep6.ipynb` (same Drive folder) and updates `model.py`, `trainer.py`, `fast_eval.py`, and `ensemble.py` (adds `MinGRUNet`, `TCNNet`, and per-model mini-batches). The full standalone version is `colab/connectome_sweep7.ipynb` on the Mac.

13 mask-weighted models (lambda 0.5, no teacher, 4 passes): 9 GRU training-dynamics runs (optimizer step size 256 / 64 / 32 sequences x backprop length 500 / 2,000 / 5,000 / 20,000 steps, plus lr 1e-3) + 2 minGRUs (`mingru_h192_l3`, `mingru_h192_l3_b32_c5000`) + 2 causal TCNs (`tcn_c64_l11`, `tcn_c128_k3_l7`). Judged on validation half B; reference sweep 5 plain_m50 = 0.6755. **Keep this tab open while it runs.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "CPUs: $(nproc)"; free -g | head -2; df -h /content | tail -1

In [ ]:
USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs'
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(OUT_DIR, exist_ok=True); print('results ->', OUT_DIR)

In [ ]:
%%bash
mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets

In [ ]:
!pip -q install onnx onnxruntime
%cd /content/wnn

In [ ]:
# sweep 6's notebook carries the base pipeline code; overwrite the files changed for sweep 7
import json, os, glob
src = None
for cand in ['/content/drive/MyDrive/wunder_challenge/connectome_sweep6.ipynb'] + glob.glob('/content/drive/MyDrive/*/wunder_challenge/connectome_sweep6.ipynb'):
    if os.path.exists(cand): src = cand; break
assert src, 'connectome_sweep6.ipynb not found in Drive'
os.makedirs('/content/wnn/pipeline', exist_ok=True); os.makedirs('/content/wnn/analysis', exist_ok=True)
n = 0
for c in json.load(open(src))['cells']:
    s = ''.join(c['source'])
    if c['cell_type'] == 'code' and s.startswith('%%writefile /content/wnn/'):
        path, _, body = s.partition('\n')
        path = path.split(' ', 1)[1].strip()
        open(path, 'w').write(body if body.endswith('\n') or not body else body + '\n'); n += 1
print(n, 'base files written from', src)
UPDATES = {'pipeline/model.py': '"""Stateful sequence models (GRU/LSTM, minGRU, causal TCN), metric-aligned losses, ONNX step export and\nbaseline import. Every model has forward(x, h, x_prev) -> (pred, new_state) for chunked training/evaluation;\nminGRU and TCN also have stream_step() for the one-row ONNX graph (see research/architectures_cnn_rnn_transformer.md)."""\nimport math\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\n\nN_FEATURES = 112\nCLIP = 2.0\n\n\ndef _model_inputs(x, x_prev, use_diff, n_extra):\n    """The 112 features, then their row-to-row diffs (use_diff), then the extra causal features (n_extra)."""\n    base = x[..., :N_FEATURES]\n    parts = [base]\n    if use_diff:\n        first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n        prev = torch.cat([first, base[:, :-1]], 1)\n        parts.append(base - prev)\n    if n_extra:\n        assert x.shape[-1] >= N_FEATURES + n_extra, "model needs the extra features as input"\n        parts.append(x[..., N_FEATURES:N_FEATURES + n_extra])\n    return torch.cat(parts, -1) if len(parts) > 1 else base\n\n\nclass GRUNet(nn.Module):\n    """Stacked GRU (or LSTM with cell="lstm") + linear head. The recurrent module keeps the\n    attribute name `gru` for either cell so older checkpoints load unchanged."""\n\n    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0, cell="gru", n_extra=0):\n        super().__init__()\n        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout, cell=cell, n_extra=n_extra)\n        self.use_diff = use_diff\n        self.cell = cell\n        self.n_extra = n_extra  # extra causal features (pipeline/features.py), appended after the 112 inputs\n        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n        rnn = {"gru": nn.GRU, "lstm": nn.LSTM}[cell]\n        self.gru = rnn(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)\n        self.head = nn.Linear(hidden, 2)\n\n    def forward(self, x, h=None, x_prev=None):\n        """x [B,T,112 (+ extras)]; h [L,B,H] or None; x_prev [B,>=112] = last row of the previous\n        chunk (None at sequence start, where the first diff is zero). Input order to the GRU:\n        the 112 features, then their diffs (use_diff), then the extras (n_extra)."""\n        base = x[..., :N_FEATURES]\n        parts = [base]\n        if self.use_diff:\n            first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n            prev = torch.cat([first, base[:, :-1]], 1)\n            parts.append(base - prev)\n        if self.n_extra:\n            assert x.shape[-1] >= N_FEATURES + self.n_extra, "model needs the extra features as input"\n            parts.append(x[..., N_FEATURES:N_FEATURES + self.n_extra])\n        out, h = self.gru(torch.cat(parts, -1) if len(parts) > 1 else base, h)\n        return self.head(out), h\n\n\ndef _g(x):\n    """minGRU candidate activation: x + 0.5 for x >= 0, sigmoid(x) below, so the candidate is always positive."""\n    return torch.where(x >= 0, x + 0.5, torch.sigmoid(x))\n\n\ndef _log_g(x):\n    return torch.where(x >= 0, torch.log(F.relu(x) + 0.5), -F.softplus(-x))\n\n\nclass MinGRUNet(nn.Module):\n    """Stack of minGRU layers (Feng et al. 2024, "Were RNNs All We Needed?") + linear head.\n\n    Per layer: z_t = sigmoid(W_z u_t), h_t = (1 - z_t) * h_{t-1} + z_t * g(W_h u_t). Gate and candidate depend\n    only on the layer input u_t (not on h_{t-1}), so training runs a parallel log-space scan over each chunk\n    (no step-by-step loop; long chunks are cheap), while inference stays an O(1) update per row with ~1/3 of a GRU\n    layer\'s multiply-adds. The nonlinear state interactions a GRU has come from stacking layers instead.\n    Gate biases start at log-spaced memory lengths of 2..2000 rows (chrono-style), so some units remember long\n    from the first step. State: one [B, H] tensor per layer."""\n    BLOCK = 128  # scan block: keeps cumulative log-gates small enough for float32 (state is carried across blocks)\n\n    def __init__(self, hidden=192, layers=3, use_diff=False, dropout=0.0, n_extra=0):\n        super().__init__()\n        self.config = dict(arch="mingru", hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout,\n                           n_extra=n_extra)\n        self.use_diff, self.n_extra, self.hidden = use_diff, n_extra, hidden\n        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n        self.proj = nn.ModuleList([nn.Linear(in_dim if l == 0 else hidden, 2 * hidden) for l in range(layers)])\n        self.drop = nn.Dropout(dropout)\n        self.head = nn.Linear(hidden, 2)\n        with torch.no_grad():  # chrono init: 1 / z = memory length tau, log-uniform in [2, 2000]\n            for p in self.proj:\n                tau = torch.exp(torch.empty(hidden).uniform_(math.log(2.0), math.log(2000.0)))\n                p.bias[:hidden] = -torch.log(tau - 1)\n\n    @classmethod\n    def _scan(cls, k, c, h0):\n        """h_t = a_t h_{t-1} + b_t with a = 1 - sigmoid(k), b = sigmoid(k) g(c), in log space per block:\n        h_t = A_t h0 + sum_{j<=t} (A_t / A_j) b_j, A_t = prod_{i<=t} a_i. k, c [B,T,H]; h0 [B,H] or None (zeros)."""\n        log_a = -F.softplus(k)\n        log_b = -F.softplus(-k) + _log_g(c)\n        outs = []\n        for s in range(0, k.shape[1], cls.BLOCK):\n            a_star = log_a[:, s:s + cls.BLOCK].cumsum(1)\n            h = torch.exp(a_star + torch.logcumsumexp(log_b[:, s:s + cls.BLOCK] - a_star, 1))\n            if h0 is not None:\n                h = h + torch.exp(a_star) * h0[:, None]\n            outs.append(h)\n            h0 = h[:, -1]\n        return torch.cat(outs, 1)\n\n    def forward(self, x, h=None, x_prev=None):\n        """x [B,T,112 (+ extras)]; h list of [B,H] per layer or None; x_prev as in GRUNet."""\n        z = _model_inputs(x, x_prev, self.use_diff, self.n_extra)\n        new = []\n        for l, proj in enumerate(self.proj):\n            k, c = proj(z).chunk(2, -1)\n            z = self._scan(k, c, None if h is None else h[l])\n            new.append(z[:, -1])\n            if l < len(self.proj) - 1:\n                z = self.drop(z)\n        return self.head(z), new\n\n    def stream_states(self):\n        return [(f"h{l}", (1, self.hidden)) for l in range(len(self.proj))]\n\n    def stream_step(self, z, states):\n        """One row: z [1,1,in] (already built inputs), states list of [1,H] -> (pred [1,1,2], new states)."""\n        z, new = z[:, 0], []\n        for proj, h in zip(self.proj, states):\n            k, c = proj(z).chunk(2, -1)\n            z = h + torch.sigmoid(k) * (_g(c) - h)\n            new.append(z)\n        return self.head(z)[:, None], new\n\n\nclass TCNNet(nn.Module):\n    """Causal dilated temporal convolution network (Bai et al. 2018, WaveNet-style) + linear head.\n\n    Input projection to `hidden` channels, then `layers` pre-norm residual blocks\n    z <- z + W_mix relu(conv_{kernel, dilation}(LayerNorm(z))), dilation = kernel**l, so the receptive field\n    is kernel**layers rows (2**11 = 2048, 3**7 = 2187). Each output sees a fixed window of past rows; unlike an\n    RNN nothing older can leak in. State between chunks: the last (kernel - 1) * dilation normalised inputs of\n    each block ([B, pad_l, C]), zeros before the sequence start, so chunked training, evaluation and row-by-row\n    inference see exactly the same context."""\n\n    def __init__(self, hidden=64, layers=11, use_diff=False, dropout=0.0, n_extra=0, kernel=2):\n        super().__init__()\n        assert kernel >= 2, "TCN kernel must be >= 2"\n        self.config = dict(arch="tcn", hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout,\n                           n_extra=n_extra, kernel=kernel)\n        self.use_diff, self.n_extra, self.hidden, self.kernel = use_diff, n_extra, hidden, kernel\n        self.dilations = [kernel ** l for l in range(layers)]\n        self.pads = [(kernel - 1) * d for d in self.dilations]\n        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n        self.inp = nn.Linear(in_dim, hidden)\n        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])\n        self.convs = nn.ModuleList([nn.Conv1d(hidden, hidden, kernel, dilation=d) for d in self.dilations])\n        self.mix = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])\n        self.drop = nn.Dropout(dropout)\n        self.out_norm = nn.LayerNorm(hidden)\n        self.head = nn.Linear(hidden, 2)\n\n    @property\n    def receptive_field(self):\n        return 1 + sum(self.pads)\n\n    def forward(self, x, h=None, x_prev=None):\n        """x [B,T,112 (+ extras)]; h list of [B, pad_l, C] per block or None; x_prev as in GRUNet."""\n        z = self.inp(_model_inputs(x, x_prev, self.use_diff, self.n_extra))\n        B, new = z.shape[0], []\n        for l, (norm, conv, mix, pad) in enumerate(zip(self.norms, self.convs, self.mix, self.pads)):\n            u = norm(z)\n            past = h[l] if h is not None else u.new_zeros(B, pad, u.shape[-1])\n            full = torch.cat([past, u], 1)  # [B, pad + T, C]\n            new.append(full[:, -pad:])\n            y = conv(full.transpose(1, 2)).transpose(1, 2)  # [B, T, C]\n            z = z + mix(self.drop(F.relu(y)))\n        return self.head(self.out_norm(z)), new\n\n    def stream_states(self):\n        return [(f"b{l}", (1, pad, self.hidden)) for l, pad in enumerate(self.pads)]\n\n    def stream_step(self, z, states):\n        """One row: z [1,1,in], states list of [1, pad_l, C] -> (pred [1,1,2], new states). Buffers are shifted\n        with Concat + Slice (simple, but copies every buffer once per row: see the notes on ring buffers)."""\n        z, new = self.inp(z), []\n        for norm, conv, mix, d, buf in zip(self.norms, self.convs, self.mix, self.dilations, states):\n            full = torch.cat([buf, norm(z)], 1)  # [1, pad + 1, C]\n            new.append(full[:, 1:])\n            taps = full[:, ::d]  # the kernel\'s inputs: rows t - (k-1)d, ..., t - d, t\n            y = F.conv1d(taps.transpose(1, 2), conv.weight, conv.bias).transpose(1, 2)  # [1, 1, C]\n            z = z + mix(F.relu(y))\n        return self.head(self.out_norm(z)), new\n\n\nARCHS = {"gru": GRUNet, "mingru": MinGRUNet, "tcn": TCNNet}\n\n\ndef build_net(config):\n    """Model from a saved config. GRU/LSTM configs predate the `arch` key, so it defaults to gru."""\n    cfg = dict(config)\n    return ARCHS[cfg.pop("arch", "gru")](**cfg)\n\n\ndef needs_extra(net):\n    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n    return max(getattr(m, "n_extra", 0) for m in members)\n\n\ndef detach_state(h):\n    """Detach a recurrent state: tensor (GRU), (h, c) tuple (LSTM) or list of those (ensemble)."""\n    if isinstance(h, torch.Tensor):\n        return h.detach()\n    return type(h)(detach_state(x) for x in h)\n\n\ndef macs(config) -> int:\n    """Multiply-adds per row (inference cost model). Ensembles: sum over members."""\n    if "members" in config:\n        return sum(macs(m) for m in config["members"])\n    arch = config.get("arch", "gru")\n    H, n_in = config["hidden"], N_FEATURES * (2 if config["use_diff"] else 1)\n    if arch == "mingru":  # one [in -> 2H] projection per layer\n        return sum(2 * H * (n_in if l == 0 else H) for l in range(config["layers"])) + 2 * H\n    if arch == "tcn":  # input projection, per block a k-tap conv + 1x1 mix (LayerNorms ignored)\n        return n_in * H + config["layers"] * (config["kernel"] + 1) * H * H + 2 * H\n    gates = 4 if config.get("cell", "gru") == "lstm" else 3\n    total = 0\n    for _ in range(config["layers"]):\n        total += gates * H * (n_in + H)\n        n_in = H\n    return total + 2 * H\n\n\nBASELINE_MACS = macs(dict(hidden=128, layers=2, use_diff=False))  # 190,720\n\n\ndef _rw(w, rw):\n    """Multiply metric weights [N,2] by optional per-row weights [N]."""\n    return w if rw is None else w * rw[:, None]\n\n\ndef wmse_loss(p, y, rw=None):\n    """Weighted MSE with the metric\'s clipping and |y| weights (times optional row weights). p, y: [N, 2]."""\n    y = y.clamp(-CLIP, CLIP)\n    w = _rw(y.abs(), rw)\n    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n\n\ndef distill_loss(p, teacher, y, weight="y", rw=None):\n    """MSE to the teacher\'s predictions. Row weights: "y" = |clipped target| (like the metric),\n    "none" = uniform, "teacher" = |teacher prediction| (no dependence on the noisy target)."""\n    if weight == "y":\n        w = y.clamp(-CLIP, CLIP).abs()\n    elif weight == "teacher":\n        w = teacher.abs()\n    elif weight == "none":\n        w = torch.ones_like(teacher)\n    else:\n        raise ValueError(weight)\n    w = _rw(w, rw)\n    return ((w * (p - teacher) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n\n\ndef wpearson(p, y, rw=None):\n    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> [2]."""\n    y = y.clamp(-CLIP, CLIP)\n    w = _rw(y.abs(), rw)\n    W = w.sum(0) + 1e-8\n    yc = y - (w * y).sum(0) / W\n    pc = p - (w * p).sum(0) / W\n    cov = (w * yc * pc).sum(0)\n    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)\n\n\ndef compute_loss(kind, p, y, alpha=0.1, rw=None):\n    if kind == "wmse":\n        return wmse_loss(p, y, rw)\n    if kind == "pearson":\n        return 1 - wpearson(p, y, rw).mean()\n    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping\n        return 1 - wpearson(p, y, rw).mean() + alpha * wmse_loss(p, y, rw)\n    raise ValueError(kind)\n\n\nclass EnsembleNet(nn.Module):\n    """Average of several GRUNets. Each member\'s output is multiplied by `scales[k]` (per target)\n    first, so members with different prediction spreads count equally. h is a list, one per member."""\n\n    def __init__(self, nets, scales):\n        super().__init__()\n        self.nets = nn.ModuleList(nets)\n        self.register_buffer("scales", torch.as_tensor(np.asarray(scales), dtype=torch.float32))  # [K, 2]\n        self.use_diff = any(n.use_diff for n in nets)\n        self.config = {"members": [n.config for n in nets], "scales": np.asarray(scales).tolist()}\n\n    def forward(self, x, h=None, x_prev=None):\n        h = h if h is not None else [None] * len(self.nets)\n        preds, new_h = [], []\n        for k, (net, hk) in enumerate(zip(self.nets, h)):\n            p, hn = net(x, hk, x_prev if net.use_diff else None)\n            preds.append(p * self.scales[k])\n            new_h.append(hn)\n        return torch.stack(preds).mean(0), new_h\n\n\ndef _split_layers(net):\n    """Single-layer copies of a stacked GRU/LSTM, so the ONNX graph gets one state input per layer\n    (as in the baseline) instead of slicing/concatenating a stacked state on every row."""\n    rnn, mods = net.gru, []\n    for l in range(rnn.num_layers):\n        m = type(rnn)(rnn.input_size if l == 0 else rnn.hidden_size, rnn.hidden_size, 1, batch_first=True)\n        for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):\n            getattr(m, f"{n}_l0").data.copy_(getattr(rnn, f"{n}_l{l}").data)\n        mods.append(m)\n    return nn.ModuleList(mods)\n\n\ndef _member_states(m):\n    """(name suffix, shape) of each streaming state of one model, in graph order."""\n    if isinstance(m, GRUNet):\n        H, lstm = m.config["hidden"], m.cell == "lstm"\n        return [s for l in range(m.config["layers"])\n                for s in [(f"h{l}", (1, 1, H))] + ([(f"c{l}", (1, 1, H))] if lstm else [])]\n    return m.stream_states()\n\n\nclass _Step(nn.Module):\n    """One-row step graph: (x [1,1,112], x_prev [1,112], *states) -> (pred [1,1,2], *new states).\n    States are flat, per member: GRU one h [1,1,H] per layer, LSTM h and c per layer, minGRU one h [1,H] per\n    layer, TCN one input buffer [1, pad, C] per block."""\n\n    def __init__(self, net):\n        super().__init__()\n        self.ensemble = isinstance(net, EnsembleNet)\n        members = list(net.nets) if self.ensemble else [net]\n        self.rnn = [isinstance(m, GRUNet) for m in members]\n        self.layers = nn.ModuleList([_split_layers(m) if r else nn.ModuleList() for m, r in zip(members, self.rnn)])\n        self.heads = nn.ModuleList([m.head if r else nn.Identity() for m, r in zip(members, self.rnn)])\n        self.streams = nn.ModuleList([nn.Identity() if r else m for m, r in zip(members, self.rnn)])\n        self.n_states = [len(_member_states(m)) for m in members]\n        self.diff = [m.use_diff for m in members]\n        self.n_extra = [getattr(m, "n_extra", 0) for m in members]\n        self.lstm = [r and m.cell == "lstm" for m, r in zip(members, self.rnn)]\n        self.register_buffer("scales", net.scales if self.ensemble else torch.ones(1, 2))\n\n    def forward(self, x, x_prev, *states):\n        states, new, preds, i = list(states), [], [], 0\n        for k, layers in enumerate(self.layers):\n            base = x[..., :N_FEATURES]\n            parts = [base] + ([base - x_prev[:, None]] if self.diff[k] else []) + \\\n                    ([x[..., N_FEATURES:N_FEATURES + self.n_extra[k]]] if self.n_extra[k] else [])\n            z = torch.cat(parts, -1) if len(parts) > 1 else base\n            if not self.rnn[k]:  # minGRU / TCN: the model\'s own one-row update\n                p, ns = self.streams[k].stream_step(z, states[i:i + self.n_states[k]])\n                new += ns\n                i += self.n_states[k]\n            else:\n                for layer in layers:\n                    if self.lstm[k]:\n                        z, (h, c) = layer(z, (states[i], states[i + 1]))\n                        new += [h, c]\n                        i += 2\n                    else:\n                        z, h = layer(z, states[i])\n                        new.append(h)\n                        i += 1\n                p = self.heads[k](z)\n            preds.append(p * self.scales[k] if self.ensemble else p)\n        pred = preds[0] if len(preds) == 1 else torch.stack(preds).mean(0)\n        return (pred, *new)\n\n\ndef state_names(net):\n    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n    return [f"m{k}_{s}" for k, m in enumerate(members) for s, _ in _member_states(m)]\n\n\ndef export_onnx(net, path: str):\n    """Export a model (GRUNet, MinGRUNet, TCNNet) or EnsembleNet as a one-row step graph. Inputs: x, x_prev\n    (dropped from the graph when no member uses diff features), then the states per member named m{k}_<state>\n    (h{l} / c{l} for GRU/LSTM and minGRU, b{l} for TCN buffers); outputs: pred, then the new states in the same\n    order. All states start at zero (solution.py zero-fills them from the graph\'s input shapes)."""\n    net = net.eval().cpu()\n    names = state_names(net)\n    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n    states = [torch.zeros(*shape) for m in members for _, shape in _member_states(m)]\n    args = (torch.zeros(1, 1, N_FEATURES + needs_extra(net)), torch.zeros(1, N_FEATURES), *states)\n    with torch.no_grad():\n        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", *names],\n                          output_names=["pred", *[n + "_out" for n in names]], opset_version=17, dynamo=False)\n\n\ndef load_baseline(onnx_path: str) -> GRUNet:\n    """Load the organisers\' 2-layer GRU ONNX weights into a GRUNet (for checks / warm starts)."""\n    import onnx\n    from onnx import numpy_helper\n\n    g = onnx.load(onnx_path).graph\n    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}\n    grus = [n for n in g.node if n.op_type == "GRU"]\n    matmul = next(n for n in g.node if n.op_type == "MatMul")\n    H = 128\n    net = GRUNet(hidden=H, layers=len(grus))\n\n    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n\n        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)\n\n    sd = {}\n    for l, n in enumerate(grus):\n        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))\n        sd[f"gru.weight_ih_l{l}"] = reorder(W)\n        sd[f"gru.weight_hh_l{l}"] = reorder(R)\n        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])\n        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])\n    sd["head.weight"] = init[matmul.input[1]].T\n    sd["head.bias"] = init["reg_head.bias"]\n    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in sd.items()})\n    return net\n', 'pipeline/trainer.py': '"""Shared training loop: one data stream feeds any number of experiments.\n\nEvery batch of whole sequences is read from parquet once and then used to train each model in\nturn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= 99).\nEvaluation is shared the same way: one pass over the eval data scores all models.\n\nOptional per model: EMA of the weights (evaluated and saved instead of the raw weights),\ndistillation towards a teacher\'s predictions on the training set (see pipeline/teacher.py), its own\nnumber of passes, and a data fold to leave out (out-of-fold teachers: sequences with\nid % n_folds == fold are skipped by that model).\n"""\nimport copy\nimport json\nimport math\nimport os\nimport shutil\nimport time\nfrom dataclasses import asdict, dataclass, field\n\nimport numpy as np\nimport torch\n\nfrom utils import WARMUP\n\nfrom .data import SeqStore, split_train\nfrom .features import N_EXTRA, offline as extra_features\nfrom .fast_eval import evaluate_many\nfrom .model import BASELINE_MACS, GRUNet, build_net, compute_loss, detach_state, distill_loss, export_onnx, load_baseline, macs\n\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n\n@dataclass\nclass ExpConfig:\n    """Per-model settings. Everything about the data stream is shared (see Common)."""\n    name: str\n    hidden: int = 128\n    layers: int = 2\n    diff: bool = False\n    dropout: float = 0.0\n    init_baseline: bool = False\n    loss: str = "wmse"          # wmse | pearson | mix\n    alpha: float = 0.1          # wmse weight in mix\n    lr: float = 2e-3\n    wd: float = 1e-4\n    seed: int = 0\n    cell: str = "gru"           # gru | lstm\n    chunk: int | None = None    # TBPTT length for this model (default: Common.chunk)\n    ema: float = 0.0            # >0: keep an EMA of the weights with this decay; eval/save the EMA\n    distill: float = 0.0        # weight of the teacher loss (0 = off, 1 = teacher only); needs a teacher\n    distill_weight: str = "y"   # teacher-loss row weights: y (|clipped target|, like the metric) | none | teacher\n    teacher: str | None = None  # this model\'s teacher .npy (default: Common.teacher)\n    epochs: int | None = None   # passes for this model (default: Common.epochs)\n    fold: int | None = None     # leave out training sequences with id % Common.n_folds == fold\n    mask_weight: float = 0.0    # lambda: row weight (1 - lambda) + lambda * p^mask_power / mean(p^mask_power)\n    mask_power: float = 1.0     # p = predicted p(scored) from Common.row_weights (pipeline/mask_model.py)\n    feats: bool = False         # also feed the 36 extra causal features (pipeline/features.py)\n    micro_batch: int | None = None  # sequences per optimizer step for this model (default: the whole data batch)\n    arch: str = "gru"           # gru (cell gru|lstm) | mingru | tcn (see pipeline/model.py)\n    kernel: int = 2             # tcn only: kernel size; dilations kernel**l, receptive field kernel**layers rows\n\n\n@dataclass\nclass Common:\n    epochs: int = 4\n    batch: int = 256            # sequences per batch\n    chunk: int = 500            # TBPTT length\n    clip_grad: float = 1.0\n    train_seqs: int | None = None\n    holdout: int = 500\n    valid_seqs: int = 400\n    evals_per_epoch: int = 2\n    seed: int = 0               # data order + holdout split\n    device: str = "auto"\n    threads: int = 4            # parquet reader threads\n    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)\n    cache_train: bool = False   # keep training sequences in RAM after first read\n    data_dir: str = "datasets"\n    out_dir: str = "runs"\n    teacher: str | None = None  # teacher predictions on train.parquet (.npy, [n_seqs, 20000, 2]) for distillation\n    n_folds: int = 4            # for ExpConfig.fold\n    row_weights: str | None = None  # p(scored) per train row (.npy [n_seqs, 20000]) for ExpConfig.mask_weight\n    valid_split: str | None = None  # split.json from mask_model: evaluate and select on validation half B\n    extra: dict = field(default_factory=dict)\n\n\nclass Experiment:\n    def __init__(self, cfg: ExpConfig, common: Common, device, n_batches):\n        self.cfg = cfg\n        self.chunk = cfg.chunk or common.chunk\n        self.epochs = cfg.epochs or common.epochs\n        self.teacher_path = (cfg.teacher or common.teacher) if cfg.distill > 0 else None\n        assert cfg.distill == 0 or self.teacher_path, f"{cfg.name}: distill > 0 needs a teacher"\n        self.micro = cfg.micro_batch or common.batch\n        total_steps = (self.epochs * n_batches * math.ceil(common.batch / self.micro)  # upper bound on steps\n                       * math.ceil(20000 / self.chunk))\n        torch.manual_seed(cfg.seed)\n        if cfg.init_baseline:\n            assert cfg.arch == "gru" and not cfg.diff and cfg.hidden == 128 and cfg.layers == 2, "baseline shape is fixed"\n            self.net = load_baseline("baseline/baseline.onnx")\n        elif cfg.arch == "gru":\n            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout, cfg.cell,\n                              n_extra=N_EXTRA if cfg.feats else 0)\n        else:\n            self.net = build_net(dict(arch=cfg.arch, hidden=cfg.hidden, layers=cfg.layers, use_diff=cfg.diff,\n                                      dropout=cfg.dropout, n_extra=N_EXTRA if cfg.feats else 0,\n                                      **({"kernel": cfg.kernel} if cfg.arch == "tcn" else {})))\n        self.net.to(device)\n        self.ema_net = copy.deepcopy(self.net) if cfg.ema > 0 else None\n        self.n_updates = 0\n        self.total_steps = total_steps\n        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)\n        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, total_steps=total_steps,\n                                                         pct_start=max(0.05, 3 / total_steps))\n        self.dir = os.path.join(common.out_dir, cfg.name)\n        os.makedirs(self.dir, exist_ok=True)\n        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")\n        self.best = -1.0\n        self.params = sum(p.numel() for p in self.net.parameters())\n        self.cost = macs(self.net.config) / BASELINE_MACS\n\n    @property\n    def eval_net(self):\n        return self.ema_net if self.ema_net is not None else self.net\n\n    @torch.no_grad()\n    def update_ema(self):\n        self.n_updates += 1\n        if self.ema_net is None:\n            return\n        decay = min(self.cfg.ema, (1 + self.n_updates) / (10 + self.n_updates))  # warm-up\n        for pe, p in zip(self.ema_net.parameters(), self.net.parameters()):\n            pe.lerp_(p, 1 - decay)\n\n    def log(self, **kw):\n        line = json.dumps(kw)\n        self.log_f.write(line + "\\n")\n        self.log_f.flush()\n        print(f"[{self.cfg.name}] {line}", flush=True)\n\n\ndef _store(common, name):\n    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), n_threads=common.threads)\n\n\ndef train_many(configs: list[ExpConfig], common: Common):\n    from .fast_eval import pick_device\n\n    device = pick_device(common.device)\n    if device.type == "cuda":\n        torch.backends.cudnn.benchmark = True\n    rng = np.random.default_rng(common.seed)\n    train, valid = _store(common, "train"), _store(common, "valid")\n    if common.cache_train:\n        train.enable_cache()\n    if common.cache_eval:\n        valid.enable_cache()\n    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)\n    if common.train_seqs:\n        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)\n    valid_B = json.load(open(common.valid_split))["B"] if common.valid_split else None\n    valid_ids = (valid_B or list(range(valid.n_seqs)))[:common.valid_seqs]\n    hold_store = train\n    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of train\n        hold_store = _store(common, "train")\n        hold_store.enable_cache()\n\n    n_batches = math.ceil(len(train_ids) / common.batch)\n    exps = [Experiment(c, common, device, n_batches) for c in configs]\n    teachers = {}  # path -> memmap, loaded once and shared by the models that use it\n    for path in sorted({e.teacher_path for e in exps if e.teacher_path}):\n        teachers[path] = np.load(path, mmap_mode="r")\n        assert teachers[path].shape[:2] == (train.n_seqs, 20000), f"teacher {path}: shape {teachers[path].shape}"\n    row_w, p_norm = None, {}\n    if any(e.cfg.mask_weight > 0 for e in exps):\n        assert common.row_weights, "mask_weight > 0 needs --row-weights (see pipeline/mask_model.py)"\n        row_w = np.load(common.row_weights, mmap_mode="r")\n        assert row_w.shape == (train.n_seqs, 20000), f"row weights shape {row_w.shape}"\n        sample = np.asarray(row_w[np.sort(np.asarray(train_ids))[::20], 99:], dtype=np.float32)\n        for pw in {e.cfg.mask_power for e in exps if e.cfg.mask_weight > 0}:\n            p_norm[pw] = float((sample ** pw).mean())  # so the average row weight stays about 1\n            print(f"row weights: mean p^{pw:g} = {p_norm[pw]:.4f}", flush=True)\n    n_epochs = max(e.epochs for e in exps)\n    t0 = time.time()\n    for e in exps:\n        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), params=e.params,\n              cost=round(e.cost, 3), train_seqs=len(train_ids), holdout=len(hold_ids), steps=e.total_steps)\n    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)\n\n    def run_eval(epoch, bi, group):\n        nets = [e.eval_net for e in group]\n        vs = evaluate_many(nets, valid, valid_ids, device)\n        hs = evaluate_many(nets, hold_store, hold_ids, device)\n        for e, v, h in zip(group, vs, hs):\n            # with a valid split: select on half B\'s scored rows (the mask model never saw them);\n            # otherwise don\'t select on the visible valid mask alone\n            sel = v["scored"] if valid_B else (v["scored"] + h["all"]) / 2\n            is_best = sel > e.best\n            if is_best:\n                e.best = sel\n                torch.save({"config": e.net.config, "state_dict": e.eval_net.state_dict(), "exp": asdict(e.cfg),\n                            "n_folds": common.n_folds},\n                           os.path.join(e.dir, "best.pt"))\n            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), valid_all=round(v["all"], 6),\n                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, time=round(time.time() - t0, 1))\n            e.net.train()\n\n    for epoch in range(n_epochs):\n        active = [e for e in exps if epoch < e.epochs]\n        for e in active:\n            e.net.train()\n        ids = np.asarray(rng.permutation(train_ids))\n        for bi, (X, Y, _) in enumerate(train.batches(list(ids), common.batch)):\n            tb = time.time()\n            batch_ids = ids[bi * common.batch:(bi + 1) * common.batch]\n            if any(e.cfg.feats for e in active):  # extras computed once per batch, shared by every model, and\n                # joined on the GPU: the joined 256-seq batch would be another 3 GB array in host RAM\n                Xd = torch.cat([torch.from_numpy(X).to(device), torch.from_numpy(extra_features(X)).to(device)], -1)\n            else:\n                Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy per batch\n            Yd = torch.from_numpy(Y).to(device, non_blocking=True)\n            order = torch.as_tensor(np.argsort(np.argsort(batch_ids)), device=device)  # memmap reads in file order\n            Td = {path: torch.from_numpy(np.asarray(t[np.sort(batch_ids)], dtype=np.float32)).to(device)[order]\n                  for path, t in teachers.items() if any(e.teacher_path == path for e in active)}\n            Pd = None\n            if row_w is not None and any(e.cfg.mask_weight > 0 for e in active):\n                Pd = torch.from_numpy(np.asarray(row_w[np.sort(batch_ids)], dtype=np.float32)).to(device)[order]\n            for e in active:\n                Xe, Ye, Te = Xd, Yd, Td.get(e.teacher_path)\n                We = None\n                if e.cfg.mask_weight > 0:\n                    lam, pw = e.cfg.mask_weight, e.cfg.mask_power\n                    We = (1 - lam) + lam * Pd.clamp(min=0) ** pw / p_norm[pw]\n                if e.cfg.fold is not None:  # out-of-fold model: drop its fold\'s sequences from the batch\n                    rows = torch.as_tensor(np.flatnonzero(batch_ids % common.n_folds != e.cfg.fold), device=device)\n                    Xe, Ye = Xd[rows], Yd[rows]\n                    Te = Te[rows] if Te is not None else None\n                    We = We[rows] if We is not None else None\n                losses = []\n                for m0 in range(0, len(Xe), e.micro):  # mini-batches of whole sequences, each its own TBPTT pass\n                    Xm, Ym = Xe[m0:m0 + e.micro], Ye[m0:m0 + e.micro]\n                    Tm = Te[m0:m0 + e.micro] if Te is not None else None\n                    Wm = We[m0:m0 + e.micro] if We is not None else None\n                    h = xp = None\n                    for s in range(0, X.shape[1], e.chunk):\n                        x, y = Xm[:, s:s + e.chunk], Ym[:, s:s + e.chunk]\n                        p, h = e.net(x, h, xp)\n                        lo = max(0, WARMUP - s)  # skip warm-up rows\n                        if lo < x.shape[1]:\n                            pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)\n                            rw = Wm[:, s:s + e.chunk][:, lo:].reshape(-1) if Wm is not None else None\n                            lam = e.cfg.distill\n                            loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha, rw) if lam < 1 else 0.0\n                            if lam > 0:\n                                tf = Tm[:, s:s + e.chunk][:, lo:].reshape(-1, 2)\n                                loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, e.cfg.distill_weight, rw)\n                            e.opt.zero_grad(set_to_none=True)\n                            loss.backward()\n                            torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)\n                            e.opt.step()\n                            e.sched.step()\n                            e.update_ema()\n                            losses.append(loss.detach())  # no GPU sync per step\n                        h = detach_state(h)\n                        xp = x[:, -1]\n                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(torch.stack(losses).mean()), 5),\n                      lr=round(e.sched.get_last_lr()[0], 6), steps=len(losses))\n            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{n_epochs}: {time.time() - tb:.1f}s "\n                  f"for {len(active)} models, elapsed {(time.time() - t0) / 60:.1f} min", flush=True)\n            del Xd, Yd, Td, Pd\n            if bi in eval_at:\n                run_eval(epoch, bi, active)\n\n    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary table\n    for e in exps:\n        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", weights_only=False)\n        e.net.load_state_dict(ck["state_dict"])\n        e.net.to(device)\n    nets = [e.net for e in exps]\n    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # uncached: 17 GB\n    full_hold = evaluate_many(nets, hold_store, hold_ids, device)\n    full_B = evaluate_many(nets, _store(common, "valid"), valid_B, device) if valid_B else [None] * len(exps)\n    rows = []\n    for e, v, h, b in zip(exps, full_valid, full_hold, full_B):\n        sub = os.path.join(e.dir, "submission")\n        os.makedirs(sub, exist_ok=True)\n        export_onnx(e.net, os.path.join(sub, "model.onnx"))\n        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, "solution.py"))\n        shutil.copy(os.path.join(HERE, "features.py"), os.path.join(sub, "wnn_features.py"))\n        res = {"valid_full": v, "holdout": h, **({"valid_B": b} if b else {})}\n        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)\n        e.log(event="done", **res)\n        rows.append({"name": e.cfg.name, "params": e.params, "cost": round(e.cost, 2), "valid_scored": round(v["scored"], 6),\n                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),\n                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4),\n                     **({"valid_B_scored": round(b["scored"], 6), "valid_B_all": round(b["all"], 6)} if b else {})})\n    rows.sort(key=lambda r: -r["valid_B_scored"] if "valid_B_scored" in r else -(r["valid_scored"] + r["holdout_all"]))\n    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:\n        for r in rows:\n            f.write(json.dumps(r) + "\\n")\n    print("\\n" + " | ".join(rows[0].keys()))\n    for r in rows:\n        print(" | ".join(str(v) for v in r.values()))\n    return rows\n', 'pipeline/fast_eval.py': '"""Batched evaluation: runs whole sequences through the network at once (seconds instead of\nthe ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.\n\n    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052\n    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport torch\n\nfrom utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator\n\nfrom .data import SeqStore, split_train\nfrom .model import build_net, load_baseline, needs_extra\n\nNEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n\n\ndef pick_device(name="auto"):\n    if name != "auto":\n        return torch.device(name)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef with_extras(X):\n    """Append the extra causal features (pipeline/features.py) to raw inputs [B,T,112]."""\n    from .features import offline\n    return np.concatenate([X, offline(X)], -1)\n\n\n@torch.no_grad()\ndef predict_batch(net, X, device, chunk=2000):\n    if needs_extra(net) and X.shape[-1] == 112:\n        X = with_extras(X)\n    B, T, _ = X.shape\n    out = np.empty((B, T, 2), np.float32)\n    h = xp = None\n    for s in range(0, T, chunk):\n        x = torch.from_numpy(X[:, s:s + chunk]).to(device)\n        p, h = net(x, h, xp)\n        xp = x[:, -1]\n        out[:, s:s + chunk] = p.float().cpu().numpy()\n    return out\n\n\ndef evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):\n    """Score several models in one pass over the data. Per model:\n    {\'scored\': WP on scored rows, \'all\': WP on all required rows, ...} (equal without a mask)."""\n    for net in nets:\n        net.eval()\n    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]\n    extra = any(needs_extra(n) for n in nets)\n    for X, Y, M in store.batches(list(ids), batch_size):\n        X = with_extras(X) if extra else X  # once per batch, shared by every model that uses them\n        for net, (acc, acc_all) in zip(nets, accs):\n            P = predict_batch(net, X, device)\n            for b in range(len(X)):\n                acc.add(Y[b], P[b], M[b])\n                if store.has_mask:\n                    acc_all.add(Y[b], P[b], NEED)\n    out = []\n    for acc, acc_all in accs:\n        r = acc.result()\n        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": r["blocks"]}\n        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else res["scored"]\n        out.append(res)\n    return out\n\n\ndef evaluate(net, store: SeqStore, ids, device, batch_size=128):\n    return evaluate_many([net], store, ids, device, batch_size)[0]\n\n\ndef load_ckpt(path):\n    ck = torch.load(path, map_location="cpu", weights_only=False)\n    net = build_net(ck["config"])\n    net.load_state_dict(ck["state_dict"])\n    return net\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    g = ap.add_mutually_exclusive_group(required=True)\n    g.add_argument("--baseline", action="store_true")\n    g.add_argument("--ckpt")\n    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")\n    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train holdout")\n    ap.add_argument("--valid-split", help="split.json from mask_model: score validation half B instead")\n    ap.add_argument("--device", default="auto")\n    args = ap.parse_args()\n\n    device = pick_device(args.device)\n    net = (load_baseline("baseline/baseline.onnx") if args.baseline else load_ckpt(args.ckpt)).to(device)\n    out = {}\n    t = time.time()\n    valid = SeqStore("datasets/valid.parquet")\n    ids = json.load(open(args.valid_split))["B"] if args.valid_split else list(range(valid.n_seqs))\n    out["valid_B" if args.valid_split else "valid"] = evaluate(net, valid, ids[:args.valid_seqs or len(ids)], device)\n    if args.holdout:\n        train = SeqStore("datasets/train.parquet")\n        _, hold = split_train(train.n_seqs, args.holdout)\n        out["holdout"] = evaluate(net, train, hold, device)\n    out["seconds"] = round(time.time() - t, 1)\n    print(json.dumps(out, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/ensemble.py': '"""Score models alone and averaged, pick an ensemble on the train holdout, optionally export it.\n\n    python -m pipeline.ensemble --ckpts "runs/*/best.pt" --budgets 1 1.5 2 --export runs/ensembles\n\nEvery model is run once over the holdout and validation data (shared loading); combinations are then\nscored from the stored predictions. Members are rescaled to a common prediction spread before\naveraging (scales measured on the holdout). Selection uses the holdout only, so the validation\nnumbers stay an honest check of the chosen combination.\n\nCost = multiply-adds per row relative to the baseline (models.macs). Real run time also has a\nfixed per-row overhead, so an ensemble\'s measured time ratio is lower than its cost; check\ncandidates with pipeline.speed. One greedy ensemble is built and exported per --budgets value.\n"""\nimport argparse\nimport glob\nimport json\nimport os\nimport shutil\nimport time\n\nimport numpy as np\n\nfrom utils import SEQUENCE_LENGTH, WARMUP, weighted_pearson\n\nfrom .data import SeqStore, split_train\nfrom .fast_eval import load_ckpt, pick_device, predict_batch, with_extras\nfrom .model import BASELINE_MACS, EnsembleNet, export_onnx, load_baseline, macs, needs_extra\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nNEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n\n\ndef load_member(path):\n    if path == "baseline":\n        return "baseline", load_baseline("baseline/baseline.onnx")\n    return os.path.basename(os.path.dirname(path)), load_ckpt(path)\n\n\ndef collect(nets, store, ids, device, masks):\n    """-> Y {mask: [N,2]}, P {mask: [K,N,2]} for masks \'all\' (required rows) and/or \'scored\'."""\n    ys = {m: [] for m in masks}\n    ps = {m: [[] for _ in nets] for m in masks}\n    extra = any(needs_extra(n) for n in nets)\n    for X, Y, M in store.batches(list(ids), 128):\n        X = with_extras(X) if extra else X\n        sel = {"all": np.broadcast_to(NEED, M.shape), "scored": M}\n        for m in masks:\n            ys[m].append(Y[sel[m]])\n        for k, net in enumerate(nets):\n            P = predict_batch(net, X, device)\n            for m in masks:\n                ps[m][k].append(P[sel[m]])\n    return ({m: np.concatenate(ys[m]) for m in masks},\n            {m: np.stack([np.concatenate(p) for p in ps[m]]) for m in masks})\n\n\ndef wp(y, p):\n    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))\n\n\ndef combine(P, scales, idx):\n    return (P[idx] * scales[idx][:, None, :]).mean(0)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--ckpts", nargs="+", required=True, help="best.pt paths/globs, or \'baseline\'")\n    ap.add_argument("--holdout", type=int, default=500)\n    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")\n    ap.add_argument("--max-members", type=int, default=5)\n    ap.add_argument("--budgets", type=float, nargs="+", default=[99.0],\n                    help="max total cost (x baseline multiply-adds); one greedy ensemble per budget")\n    ap.add_argument("--export", help="root folder: writes <root>/cost<budget>/submission per budget")\n    ap.add_argument("--device", default="auto")\n    ap.add_argument("--threads", type=int, default=4)\n    ap.add_argument("--valid-split", help="split.json from mask_model: report on validation half B")\n    ap.add_argument("--select-on", choices=["holdout", "validA"], default="holdout",\n                    help="what greedy selection maximises: holdout all-rows WP, or validation half A scored rows "\n                         "(needs --valid-split; right for mask-weighted models, keeps half B honest)")\n    args = ap.parse_args()\n    assert args.select_on == "holdout" or args.valid_split, "--select-on validA needs --valid-split"\n\n    paths = []\n    for c in args.ckpts:\n        paths += [c] if c == "baseline" else sorted(glob.glob(c))\n    names, nets = zip(*[load_member(p) for p in paths])\n    device = pick_device(args.device)\n    nets = [n.eval().to(device) for n in nets]\n    print(f"{len(nets)} models: {\', \'.join(names)}", flush=True)\n\n    t = time.time()\n    train = SeqStore("datasets/train.parquet", args.threads)\n    valid = SeqStore("datasets/valid.parquet", args.threads)\n    _, hold_ids = split_train(train.n_seqs, args.holdout)\n    Yh, Ph = collect(nets, train, hold_ids, device, ["all"])\n    if args.valid_split:\n        split = json.load(open(args.valid_split))\n        Ya, Pa = collect(nets, valid, split["A"], device, ["scored"])\n        Yv, Pv = collect(nets, valid, split["B"], device, ["scored", "all"])  # report on half B\n    else:\n        Yv, Pv = collect(nets, valid, range(args.valid_seqs or valid.n_seqs), device, ["scored", "all"])\n    print(f"predictions collected in {time.time() - t:.0f}s", flush=True)\n    SEL = args.select_on\n\n    sd = Ph["all"].std(axis=1)  # [K, 2]\n    scales = sd.mean(0, keepdims=True) / sd\n\n    def sel_score(idx):\n        if SEL == "validA":\n            return wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n        return wp(Yh["all"], combine(Ph["all"], scales, idx))\n\n    def score(idx):  # valid_* = half B when --valid-split is given\n        out = {"holdout": wp(Yh["all"], combine(Ph["all"], scales, idx)),\n               "valid_scored": wp(Yv["scored"], combine(Pv["scored"], scales, idx)),\n               "valid_all": wp(Yv["all"], combine(Pv["all"], scales, idx))}\n        if args.valid_split:\n            out["validA"] = wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n        return out\n\n    cost = np.array([macs(n.config) / BASELINE_MACS for n in nets])\n    rows = []\n    singles = [(k, score([k])) for k in range(len(nets))]\n    for k, sc in singles:\n        rows.append({"combo": names[k], "n": 1, "cost": cost[k], **sc})\n    order = [k for k, sc in sorted(singles, key=lambda ks: -ks[1][SEL])]\n    for top in range(2, min(len(nets), args.max_members) + 1):\n        rows.append({"combo": f"top{top} by {SEL}", "n": top, "cost": cost[order[:top]].sum(),\n                     "members": [names[k] for k in order[:top]], **score(order[:top])})\n\n    def greedy(budget):\n        """Forward selection on SEL (no repeats) within a total cost budget."""\n        ok = [k for k in order if cost[k] <= budget]\n        if not ok:\n            return []\n        chosen, best = [ok[0]], singles[ok[0]][1][SEL]\n        while len(chosen) < args.max_members:\n            cand = [(k, sel_score(chosen + [k])) for k in range(len(nets))\n                    if k not in chosen and cost[chosen + [k]].sum() <= budget]\n            if not cand:\n                break\n            k, sc = max(cand, key=lambda c: c[1])\n            if sc <= best:\n                break\n            chosen.append(k)\n            best = sc\n        return chosen\n\n    picks = {}\n    for b in args.budgets:\n        chosen = greedy(b)\n        if chosen:\n            picks[b] = chosen\n            rows.append({"combo": f"greedy, cost <= {b:g}", "n": len(chosen), "cost": cost[chosen].sum(),\n                         "members": [names[k] for k in chosen], **score(chosen)})\n\n    vl = "B" if args.valid_split else ""\n    # diversity: members that disagree with the best single model are the ones an average (or a teacher) gains from\n    best_k = order[0]\n    print(f"\\nprediction correlation with {names[best_k]} (valid{vl} scored rows, mean of t0/t1):")\n    for k in order:\n        r = np.mean([np.corrcoef(Pv["scored"][k][:, j], Pv["scored"][best_k][:, j])[0, 1] for j in range(2)])\n        print(f"  {names[k][:40]:40s} {r:.4f}")\n    print(f"\\n{\'combo\':30s} {\'n\':>2s} {\'cost\':>5s} {\'holdout\':>8s} {\'valid\'+vl+\'_sc\':>9s} {\'valid\'+vl+\'_all\':>10s}"\n          + (f" {\'validA_sc\':>9s}" if args.valid_split else ""))\n    for r in rows:\n        print(f"{r[\'combo\'][:30]:30s} {r[\'n\']:2d} {r[\'cost\']:5.2f} {r[\'holdout\']:8.5f} {r[\'valid_scored\']:9.5f} "\n              f"{r[\'valid_all\']:10.5f}" + (f" {r[\'validA\']:9.5f}" if "validA" in r else "")\n              + (f"   {\'+\'.join(r[\'members\'])}" if "members" in r else ""))\n\n    if args.export:\n        for b, chosen in picks.items():\n            out = os.path.join(args.export, f"cost{b:g}", "submission")\n            os.makedirs(out, exist_ok=True)\n            net = nets[chosen[0]] if len(chosen) == 1 else EnsembleNet([nets[k] for k in chosen], scales[chosen])\n            export_onnx(net, os.path.join(out, "model.onnx"))\n            shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(out, "solution.py"))\n            shutil.copy(os.path.join(HERE, "features.py"), os.path.join(out, "wnn_features.py"))\n            row = next(r for r in rows if r["combo"] == f"greedy, cost <= {b:g}")\n            json.dump({"members": [paths[k] for k in chosen], "scales": scales[chosen].tolist(),\n                       "scores": {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in row.items()}},\n                      open(os.path.join(out, "ensemble.json"), "w"), indent=1)\n            print(f"exported cost<={b:g}: {len(chosen)} model(s) -> {out}")\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/test_export.py': '"""Check that exported ONNX step graphs + solution.py reproduce PyTorch predictions row by row.\n\n    python -m pipeline.test_export            # random GRU / diff-GRU / LSTM / minGRU / TCN / mixed ensembles\n"""\nimport importlib.util\nimport os\nimport shutil\nimport tempfile\n\nimport numpy as np\nimport pyarrow.parquet as pq\nimport torch\n\nfrom utils import FEATURE_COLUMNS, WARMUP, DataPoint\n\nfrom .fast_eval import predict_batch\nfrom .features import N_EXTRA\nfrom .model import EnsembleNet, GRUNet, MinGRUNet, TCNNet, export_onnx\n\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n\ndef run_rows(net, X):\n    d = tempfile.mkdtemp()\n    export_onnx(net, os.path.join(d, "model.onnx"))\n    shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(d, "solution.py"))\n    shutil.copy(os.path.join(HERE, "features.py"), os.path.join(d, "wnn_features.py"))\n    spec = importlib.util.spec_from_file_location("sol_test", os.path.join(d, "solution.py"))\n    mod = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(mod)\n    m = mod.PredictionModel()\n    out = np.zeros((len(X), 2), np.float32)\n    for seq in (0, 1):  # two sequences: also checks the state reset\n        for t in range(len(X)):\n            p = m.predict(DataPoint(seq, t, t >= WARMUP, X[t]))\n            if p is not None:\n                out[t] = p\n    shutil.rmtree(d)\n    return out\n\n\ndef main():\n    torch.manual_seed(0)\n    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, columns=list(FEATURE_COLUMNS))\n    X = np.column_stack([tbl[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)[:3000]\n    a, b, c = GRUNet(96), GRUNet(64, use_diff=True), GRUNet(80, layers=3, cell="lstm")\n    d, e = GRUNet(64, n_extra=N_EXTRA), GRUNet(48, use_diff=True, n_extra=N_EXTRA)\n    # minGRU: 3000 rows cross several scan blocks and chunks; TCN 3**6 = 729-row receptive field crosses chunks\n    f, g = MinGRUNet(64, layers=3), MinGRUNet(48, layers=2, use_diff=True, n_extra=N_EXTRA)\n    t1, t2 = TCNNet(32, layers=6, kernel=3), TCNNet(24, layers=8, kernel=2, use_diff=True, n_extra=N_EXTRA)\n    nets = {"gru": a, "gru_diff": b, "lstm_3layer": c, "gru_feats": d, "gru_diff_feats": e,\n            "mingru": f, "mingru_diff_feats": g, "tcn_k3": t1, "tcn_k2_diff_feats": t2,\n            "ensemble": EnsembleNet([a, b, c], np.array([[1.0, 1.1], [0.9, 1.2], [1.3, 0.7]])),\n            "ensemble_feats": EnsembleNet([a, d, e], np.array([[1.0, 1.0], [0.8, 1.2], [1.1, 0.9]])),\n            "ensemble_mixed": EnsembleNet([a, f, t1, t2], np.array([[1.0, 1.0], [0.8, 1.2], [1.1, 0.9], [0.9, 1.0]]))}\n    for name, net in nets.items():\n        net.eval()\n        ref = predict_batch(net, X[None], torch.device("cpu"), chunk=700)[0]\n        got = run_rows(net, X)\n        err = np.abs(ref[WARMUP:] - got[WARMUP:]).max()\n        print(f"{name:12s} max |torch - onnx| = {err:.2e}  {\'OK\' if err < 1e-4 else \'MISMATCH\'}")\n\n\nif __name__ == "__main__":\n    main()\n'}
for rel, body in UPDATES.items():
    open(f'/content/wnn/{rel}', 'w').write(body)
import importlib, sys; sys.path.insert(0, '/content/wnn')
from pipeline.trainer import ExpConfig
from pipeline.model import MinGRUNet, TCNNet
assert 'arch' in ExpConfig.__dataclass_fields__ and 'micro_batch' in ExpConfig.__dataclass_fields__
print('updated pipeline loaded: GRUNet + MinGRUNet + TCNNet + micro_batch')

In [ ]:
!python -m pipeline.fast_eval --baseline --valid-seqs 300

In [ ]:
MASK, CACHE = f'{OUT_DIR}/mask_model', f'{OUT_DIR}/cache'
ROWW, SPLIT = '/content/wnn/mask_train.npy', f'{OUT_DIR}/mask_model/split.json'
!cp {CACHE}/mask_train.npy {ROWW} && ls -la {ROWW}

In [ ]:
import json
experiments = [{'name': 'b256_c500', 'mask_weight': 0.5},
 {'name': 'b64_c500', 'mask_weight': 0.5, 'micro_batch': 64},
 {'name': 'b32_c500', 'mask_weight': 0.5, 'micro_batch': 32},
 {'name': 'b256_c2000', 'mask_weight': 0.5, 'chunk': 2000},
 {'name': 'b64_c2000', 'mask_weight': 0.5, 'micro_batch': 64, 'chunk': 2000},
 {'name': 'b32_c2000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 2000},
 {'name': 'b32_c5000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 5000},
 {'name': 'b32_c20000', 'mask_weight': 0.5, 'micro_batch': 32, 'chunk': 20000},
 {'name': 'b32_c500_lr1e-3', 'mask_weight': 0.5, 'micro_batch': 32, 'lr': 0.001},
 {'name': 'mingru_h192_l3', 'mask_weight': 0.5, 'arch': 'mingru', 'hidden': 192, 'layers': 3},
 {'name': 'mingru_h192_l3_b32_c5000',
  'mask_weight': 0.5,
  'arch': 'mingru',
  'hidden': 192,
  'layers': 3,
  'micro_batch': 32,
  'chunk': 5000},
 {'name': 'tcn_c64_l11',
  'mask_weight': 0.5,
  'arch': 'tcn',
  'hidden': 64,
  'layers': 11,
  'kernel': 2,
  'chunk': 2000},
 {'name': 'tcn_c128_k3_l7',
  'mask_weight': 0.5,
  'arch': 'tcn',
  'hidden': 128,
  'layers': 7,
  'kernel': 3,
  'chunk': 2000}]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')

In [ ]:
SWEEP = f'{OUT_DIR}/sweep7'
!python -m pipeline.multi_train --config experiments.json --epochs 4 --evals-per-epoch 1 --threads 8 --cache-eval --row-weights {ROWW} --valid-split {SPLIT} --out-dir {SWEEP} 2>&1 | grep -v '"event": "batch"'

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in open(f'{SWEEP}/summary.jsonl')]
df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
display(df[['name', 'cost', 'valid_B_scored', 'valid_B_all', 'valid_scored', 'holdout_all']].sort_values('valid_B_scored', ascending=False))
!python -m pipeline.ensemble --ckpts "{SWEEP}/*/best.pt" --valid-split {SPLIT} --select-on validA --threads 8 --max-members 8 --budgets 1.5 99 2>&1 | grep -vE 'Warning|_generic_rnn|warnings.warn'
!python -m pipeline.speed {SWEEP}/b256_c500/submission {SWEEP}/mingru_h192_l3/submission {SWEEP}/tcn_c64_l11/submission